In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/o4_mini_high/checkpoints/post_cell_8_try_2.pickle

trying: ['data']
me:  10
trying: ['warnings']
me:  0
trying: ['sample_discourse_id']
me:  3
trying: ['ax1']
me:  8
trying: ['np']
me:  0
trying: ['ax2']
me:  8
trying: ['stopwords']
me:  0
trying: ['agg']
me:  8
trying: ['plt']
me:  0
trying: ['train']
me:  4
trying: ['tqdm']
me:  0
trying: ['FuncFormatter']
me:  0
trying: ['style']
me:  0
trying: ['cols_to_display']
me:  4
trying: ['pd']
me:  0
trying: ['ax']
me:  9
trying: ['total_essays']
me:  9
trying: ['orig_output']
me:  2
trying: ['perc']
me:  9
trying: ['print_discourse_info']
me:  6
trying: ['pred_len']
me:  7
trying: ['pred_list']
me:  7
trying: ['pred_str']
me:  7
trying: ['sample_submission']
me:  1
trying: ['CountVectorizer']
me:  0
trying: ['glob']
me:  0
trying: ['test_txt']
me:  1
trying: ['train_txt']
me:  1
Declaring variable warnings
Declaring variable np
Declaring variable stopwords
Declaring variable plt
Declaring variable tqdm
Declaring variable FuncFormatter
Declaring variable style
Declaring variable pd
Declarin

In [4]:
%%RecordEvent
%%time
### cell 9 ###
# 1) For each essay id, grab its first and last discourse_type
disp_per_id = train.groupby('id', sort=False)["discourse_type"].agg(["first", "last"])
# 2) Number of unique essays
n_ids = train.id.nunique()

# 3) Stack into a long table so we can count first vs last in one go
disp_long = (
    disp_per_id
      .reset_index(drop=True)
      .melt(value_vars=["first","last"],
            var_name="position",
            value_name="discourse_type")
)

# 4) Count how many times each discourse_type appears in 'first' and in 'last'
counts = (
    disp_long
      .groupby(["discourse_type","position"])  
      .size()
      .unstack(fill_value=0)
)
# 5) Rename columns to match the original
counts = counts.rename(columns={"first":"counts_first", "last":"counts_last"})

# 6) Compute percents (round to 2 decimals)
counts['percent_first'] = (counts['counts_first'] / n_ids).round(2)
counts['percent_last']  = (counts['counts_last']  / n_ids).round(2)

# 7) Keep only those types that appeared first at least once
counts = counts[counts['counts_first'] > 0]

# 8) For types that never appeared last, set counts_last & percent_last to NaN
#    (mimics the left‐merge behavior)
counts['counts_last']   = counts['counts_last'].astype(float).mask(counts['counts_last'] == 0)
counts['percent_last']  = counts['percent_last'].mask(counts['counts_last'].isnull())

# 9) Reset index and sort by counts_first DESC, then discourse_type ASC to match value_counts tie-breaking
result = (
    counts
      .reset_index()
      .sort_values(by=['counts_first','discourse_type'], ascending=[False, True])
      .reset_index(drop=True)
)

# 10) Select/finalize columns
train_first_last = result[[
    'discourse_type',
    'counts_first','percent_first',
    'counts_last','percent_last'
]]

train_first_last

CPU times: user 252 ms, sys: 72 ms, total: 324 ms
Wall time: 331 ms


position,discourse_type,counts_first,percent_first,counts_last,percent_last
0,Lead,83,0.77,<NA>,<NA>
1,Position,24,0.22,5.0,0.05
2,Claim,1,0.01,2.0,0.02


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/o4_mini_high/checkpoints/post_cell_9_try_7.pickle

migration speed (bps): 719517283.8761672
---------------------------
variables to migrate:
disp_long 48
counts 48
pred_list 408
pred_str 171
print_discourse_info 144
pred_len 28
warnings 72
pd 72
disp_per_id 48
result 48
test_txt 120
train_txt 126104
ax2 48
sample_discourse_id 32
agg 48
train 48
ax1 48
glob 144
n_ids 28
np 72
stopwords 48
plt 72
sample_submission 48
CountVectorizer 1072
tqdm 1072
FuncFormatter 1072
cols_to_display 152
style 72
perc 48
ax 48
total_essays 28
train_first_last 48
orig_output 16
data 48
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/o4_mini_high/checkpoints/post_cell_9_try_7.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['train']
Intermediate variables ['train_txt', 'sample_submission', 'test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['sample_discourse_id']
Intermediate variables []
Future variables ['train']
Modified dataframes
======= Cell 2 =======
Input variables []
Active variables ['train', 'cols_to_display']
Intermediate variables []
Future variables ['sample_discourse_id']
Modified dataframes
  train
    Input columns: set()
    Changed columns: set()
    Created columns: {'discourse_len', 'pred_len'}
    Deleted columns: set()
======= Cell 3 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['train', 'sample_discourse_id']
Modified dataframes
======= Cell 4 =======
Input variables []
Active variables []
Intermediate variables []
Future variables ['train', 'sample_discourse_id']
Modified dataframes
======= Cell 5 =======
Input variabl

In [7]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_9_try_7.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[9], f)


In [ ]:
opt_output = Out.get(4)

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_9.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output
